# 29 — Choosing a more specific Test A (pre-registration addendum 3)

Estimates the four candidate specifications fixed in `report/preregistration_2026_general_addendum_3.md` (committed 27 Sept 2026 before estimation) on the Addendum 2 validation cases, and applies the pre-committed choice rule mechanically:

1. **Sensitivity:** target-swing coefficient > 0 with a 95% interval excluding 0 for Brown−Cordray (OH 2018) and Baldwin−Evers (WI 2018).
2. **Specificity (placebo):** |2008→12 coefficient × its SD| < |target coefficient × its SD| for Brown and Baldwin. For S0 the placebo coefficient comes from a separate regression, as in Addendum 2.
3. **Specificity (ordinary pairs):** target coefficient within ±0.1 for the three Michigan 2018 down-ballot pairs.

The simplest passing specification (S1, then S2, then S3) is chosen; S3 applies to Michigan only (criterion 3). Place characteristics: ACS 2016–2020 (`acs_place_controls_pull.py`).

In [1]:
import numpy as np, pandas as pd, statsmodels.formula.api as smf, re, sys, json, warnings
warnings.filterwarnings('ignore'); pd.set_option('display.width', 230)
sys.path.insert(0, '../src/data_collection'); import michigan_pull as mp
RAW = '../data/raw/'
# reuse notebook 28's frames (same cleaning, no duplication of logic)
nb28 = json.load(open('28_testA_validation.ipynb', encoding='utf-8'))
src = '\n'.join(''.join(c['source']) for c in nb28['cells'] if c['cell_type'] == 'code').replace("res.to_csv('../outputs/tables/testA_validation_addendum2.csv', index=False)", '')
g = {}; exec(src, g); m, cases = g['m'].copy(), g['cases']
def acs_cols(d):
    d = d.copy(); d['log_pop'] = np.log(d['pop'].clip(lower=1)); d['log_income'] = np.log(d.median_income); return d
# Michigan: match ACS county subdivisions to municipalities by county + normalized name, then bare name
a = acs_cols(pd.read_csv(RAW + 'validation/acs_place_controls_mi_cousub.csv', dtype={'county': str, 'county subdivision': str}))
a['county_name'] = a.NAME.str.split(', ').str[1].str.replace(' County', '', regex=False).str.replace('.', '', regex=False).str.strip()
a['muni'] = a.NAME.str.split(', ').str[0].map(mp.municipality)
m['county_n'] = m.county.str.replace('.', '', regex=False).str.replace(r"(?i)'s$", '', regex=True).str.strip()
a = a[a['pop'] > 0]
keep = ['log_pop', 'ba_share', 'log_income', 'white_nh_share']
x1 = m.merge(a[['county_name', 'muni'] + keep], left_on=['county_n', 'muni'], right_on=['county_name', 'muni'], how='left')
miss = x1[keep[0]].isna()
a['bare'] = a.muni.map(lambda s: mp._bare(s, False)); m['bare'] = m.muni.map(lambda s: mp._bare(s, False))
a_u = a[~a.duplicated(['county_name', 'bare'], keep=False)]
fill = m.loc[miss.values].merge(a_u[['county_name', 'bare'] + keep], left_on=['county_n', 'bare'], right_on=['county_name', 'bare'], how='left')
x1.loc[miss.values, keep] = fill[keep].values
m = x1
print('Michigan municipalities with ACS controls:', int(m[keep[0]].notna().sum()), 'of', len(m), '| vote share covered:', round(m.loc[m[keep[0]].notna(), 'gov_tot'].sum() / m.gov_tot.sum(), 4))
print('unmatched (largest):', m[m[keep[0]].isna()].sort_values('gov_tot', ascending=False)[['county', 'muni', 'gov_tot']].head(6).values.tolist())
ca = acs_cols(pd.read_csv(RAW + 'validation/acs_place_controls_counties.csv', dtype={'state': str, 'county': str}))
ca['fips'] = ca.state + ca.county
cases = [(chk, lab, f.merge(ca[['fips'] + keep], on='fips', how='left'), prim) for chk, lab, f, prim in cases]
for chk, lab, f, prim in cases: print(lab, 'counties with ACS:', int(f[keep[0]].notna().sum()), 'of', len(f))

1523 municipalities
OH 88 counties; unmatched: []
WI 72 counties; unmatched: []
PA 67 counties; unmatched: []
  OH 2018 Brown (Sen) - Cordray (Gov): statewide Senate D 2,358,508  Gov D 2,070,046
  WI 2018 Baldwin (Sen) - Evers (Gov): statewide Senate D 1,472,914  Gov D 1,324,307
  PA 2022 Fetterman (Sen) - Shapiro (Gov): statewide Senate D 2,751,012  Gov D 3,031,137
                                                check      role                                sample       swing    N  mean_gap  slope     lo     hi     p  outside_pm_0.1
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   primary                                   all swing_12_16 1423    -3.531 -0.085 -0.113 -0.057 0.000           False
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   primary excl. Dearborn/Dearborn Hts/Hamtramck swing_12_16 1420    -3.556 -0.084 -0.112 -0.055 0.000           False
              V1a MI 2018 Nessel (AG) - Whitmer (Gov)   placebo                                   all swing_08_12 1

In [2]:
def wsd(x, w):
    mu = np.average(x, weights=w); return float(np.sqrt(np.average((x - mu) ** 2, weights=w)))
def run(df, gap, target, controls, wcol, fe=None, placebo='swing_08_12'):
    cols = [gap, target, wcol] + controls + ([fe] if fe else []) + ([placebo] if placebo not in controls else [])
    x = df[list(dict.fromkeys(cols))].replace([np.inf, -np.inf], np.nan).dropna(); x = x[x[wcol] >= 200]
    rhs = ' + '.join([target] + controls + ([f'C({fe})'] if fe else []))
    r = smf.wls(f'{gap} ~ {rhs}', data=x, weights=x[wcol]).fit(cov_type='HC1'); ci = r.conf_int().loc[target]
    if placebo in controls: pb = r.params[placebo]
    else: pb = smf.wls(f'{gap} ~ {placebo}', data=x, weights=x[wcol]).fit(cov_type='HC1').params[placebo]
    return {'N': len(x), 'target_coef': r.params[target], 'lo': ci[0], 'hi': ci[1],
            'target_per_sd': r.params[target] * wsd(x[target], x[wcol]), 'placebo_per_sd': pb * wsd(x[placebo], x[wcol])}
ACS = ['log_pop', 'ba_share', 'log_income', 'white_nh_share']
SPEC_MI = {'S0': [], 'S1': ['swing_08_12', 'rep2012'], 'S2': ['swing_08_12', 'rep2012'] + ACS, 'S3': ['swing_08_12', 'rep2012'] + ACS}
rows = []
for lab, a_, b_, w in [('Nessel - Whitmer', 'ag_sh', 'gov_sh', 'gov_tot'), ('Benson - Whitmer', 'sos_sh', 'gov_sh', 'gov_tot'), ('Nessel - Benson', 'ag_sh', 'sos_sh', 'gov_tot'),
                       ('Stabenow - Whitmer (familiarity)', 'sen_sh', 'gov_sh', 'sen_tot')]:
    d = m.assign(gap=m[a_] - m[b_])
    for s, ctl in SPEC_MI.items():
        rows.append({'case': f'MI 2018 {lab}', 'spec': s, **run(d, 'gap', 'swing_12_16', ctl, w, fe='county' if s == 'S3' else None)})
for chk, lab, f, prim in cases:
    earlier = ['swing_08_12'] + (['swing_12_16'] if prim == 'swing_16_20' else [])
    level = 'rep2016' if prim == 'swing_16_20' else 'rep2012'
    for s, ctl in {'S0': [], 'S1': earlier + [level], 'S2': earlier + [level] + ACS}.items():
        rows.append({'case': lab, 'spec': s, **run(f, 'gap', prim, ctl, 'sen_tot')})
res = pd.DataFrame(rows); print(res.round(3).to_string(index=False))
res.to_csv('../outputs/tables/testA_specification_choice_addendum3.csv', index=False)

                                    case spec    N  target_coef     lo     hi  target_per_sd  placebo_per_sd
                MI 2018 Nessel - Whitmer   S0 1423       -0.085 -0.113 -0.057         -0.559          -0.751
                MI 2018 Nessel - Whitmer   S1 1423       -0.064 -0.079 -0.049         -0.420           0.010
                MI 2018 Nessel - Whitmer   S2 1398       -0.022 -0.057  0.013         -0.146           0.067
                MI 2018 Nessel - Whitmer   S3 1398       -0.069 -0.094 -0.044         -0.453          -0.069
                MI 2018 Benson - Whitmer   S0 1423        0.000 -0.013  0.014          0.003          -0.365
                MI 2018 Benson - Whitmer   S1 1423        0.008 -0.006  0.021          0.050          -0.253
                MI 2018 Benson - Whitmer   S2 1398        0.037  0.012  0.062          0.242          -0.105
                MI 2018 Benson - Whitmer   S3 1398        0.001 -0.018  0.020          0.006          -0.145
                 MI

## Applying the pre-committed choice rule

In [3]:
def get(case, spec): return res[(res.case == case) & (res.spec == spec)].iloc[0]
pos = ['OH 2018 Brown (Sen) - Cordray (Gov)', 'WI 2018 Baldwin (Sen) - Evers (Gov)']
pairs = ['MI 2018 Nessel - Whitmer', 'MI 2018 Benson - Whitmer', 'MI 2018 Nessel - Benson']
verdict = []
for s in ['S0', 'S1', 'S2']:
    c1 = all(get(c, s).lo > 0 for c in pos)
    c2 = all(abs(get(c, s).placebo_per_sd) < abs(get(c, s).target_per_sd) for c in pos)
    c3 = all(abs(get(c, s).target_coef) <= 0.1 for c in pairs)
    verdict.append({'spec': s, 'sensitivity': c1, 'placebo_specificity': c2, 'ordinary_pairs': c3, 'passes': c1 and c2 and c3})
s3_c3 = all(abs(get(c, 'S3').target_coef) <= 0.1 for c in pairs)
verdict = pd.DataFrame(verdict); print(verdict.to_string(index=False)); print('S3 (Michigan, criterion 3):', s3_c3)
passing = [v for v in ['S1', 'S2'] if verdict.set_index('spec').loc[v, 'passes']]
choice = passing[0] if passing else None
mi_choice = 'S3' if (choice and s3_c3) else choice
print('\nCHOSEN secondary specification:', choice, '| Michigan version:', mi_choice)
pd.DataFrame([{'chosen': choice, 'michigan_version': mi_choice}]).to_csv('../outputs/tables/testA_specification_chosen.csv', index=False)

spec  sensitivity  placebo_specificity  ordinary_pairs  passes
  S0         True                False            True   False
  S1         True                 True            True    True
  S2        False                False            True   False
S3 (Michigan, criterion 3): True

CHOSEN secondary specification: S1 | Michigan version: S3
